# mf_setup.py, the backbone of `mf6lab` for simulations with Modflow in Python

The original `mflab` was built on Matlab. Because I have no access to Matlab since my emeritat,
I switched to `Python` in 2016. Python is free and, therefore, generally accessible to students
as well as to anyone interested in programming.

This `mf_setup.py` is used in every project and case and, therefore, is resides in `mf6lab/src`. Local `mf_run_py` which imports it and runs its funcion `mf_setup()` to provide a basic setup for the model.

The general directory structure of `mf6lab` can be found in `mf6lab/src/mf6tools.py`. Its class `Dirs()` is used to set the namespace for the case so that all its directories can be readily accessed without hard-wired naming.

for every case 5 local files are required:

in the local directory `mf6lab/src/Projects/<project_name>/cases/<case_name>/src`:

1) mf_adapt.py
2) setting.py
3) mf_analyze.py
4) mf_run.py
5) <case_name>.xlsx

In the local data directory (in older projects the src directory) should hold this the Excel workbook

`mf6lab/src/Projects/<project_name>/cases/<case_name>/data`:


## The Excel workbook `<case_name>.xlsx`

The Excel workbook has a number of spreasheet that define parameters and specifies stress periods and layer properties. The spreadsheets are:

`NAM`, `SIM6`, `GWF6`, `GWT6`, `PER`, `LAY`

### The sheet `NAM`

Shows all possible packages of the groundwater flow and the groundwater transport models of Modflow 6 in the form of `Gwf???` and `Gwt???` in which Gwf stands for `Ground Water FLow` and `Gwt` stands for `Ground Water Transport` and where `???` is the mostly 3-character acronym of the package. So we get package names like `Gwfdis`, `Gwfwel`, `Gwfchd` etc as well as `Gwtdis` `Gwtadv` etc. One designates a package to be using in a given simulation by selecting it. Is is done by placing a 1 instead of a 0 in column `ON/OFF`. Selected packages are indicted in the column `Indicator` (column C). Selection of at least one package starting with `Gwf` implies that the Modflow flow model will be run and selecting at least one packages starting with `Gwt` ensures that the groundwater transport model will be run. Any combination will cause both models to be run and data to be exchanged between them.

### The sheets `SIM6`, `GWF6` `GWT6`

These three spreadsheets in the workbook contain all possible parameters for the modflow 6 `simulation`, its `Groundwater Flow Model` and its `Groundwater Transport Model` with default values. If desired they can be adapted there to the needs of the current case, which is often done for the settings of the solver. But mostly they are left along and overwritten in `mf_adapt.py`, which is imported by `mf_setup.py`.

### The sheet `MP7`

Similarly the sheet `MF7` holds all possible parameters for `Modflpath 7`, which is a program separate from `Modflow 6`.

### The sheet `PER`

This sheet is used to define the stress periods for the current case. It has columns named

`IPER`, `PERLEN`, `NSTP` and `TSMULT`

It will be read in as a pd.DataFrame to make its data available within `mf_adapt.py`.

Note that the stress period number `IPER` is zero based.

One can add as many columns as one likes, for instance one with the data at which each stress period begins.
Of course, it is not necessary to use this sheet, but then the stress periods must be defined by the user in `mf_adapt.py`. Defining them in this spreadsheet has been proven to be convenient.

Missing stress period (gaps in `IPER`) are interpreted as having the same properties as the last one that was defined (= downward filling). The last stress period must always be given as well as the first one. As a convenience, if only the last stress period is specified, then it is assumed that all previous ones have the same properties.

### The sheet `LAY`

This spreadsheet allows specifying layer properties, but of course only those that are constant throughout each layer. Properties that vary throughout the model, mostly layer elevations cannot be specified in a spreadsheet and must be specified by the user in `mf_adapt.py`.

The column heading of the `LAY` sheet will look like this

`LAYER`, `ICELLTYPE`, `k`, `k33`, `Sy`, `Ss`

One may add as many columns as one likes and also leave columns out or ignore them when setting up the model in `mf_adapt.py`. The sheet will be read into a pd.DataFrame and its data are available in `mf_adapt.py`.

The layer numbers are zero-based. Missing layers will be filled in from below. That is, the properties of missing layers are assumed to be the same as the next layer that defines them. Therefore, to specify the same properties for `n` layers all with the same properties one just needs one line defining the properties for layerin `n-1`. That layer number implies how many layers the model wil have (`n`)

## mf_adapt.py and mf_settings.py

The files mf_adapt.py and mf_settings.py are local, i.e. they are in `mf6lab/Projects/<project_name>/cases/<case_name>/src`. `mf_run.py` imports `mf_setup.py` to load which packages and modelt to use, together with the parameter. `mf_setup.py` imports `mf_adapt.py` which adapts the variables of the packages to match the actual case. `settings.py` is in turn imported by `mf_adapt.py` to set case-specific properties. `settings.py` mainly minimizes clutter in `mf_adapt.py`, and could be left out. `While` local script `mf_run.py` and `mf6lab`-wide `mf_setup_py` will always be the same for any project and case, the four local files named above are specific to each case. There structure will be more or less the same between most projects, but some adaptations will generally have to be made between cases.

When starting a new Project, create a folder with the desired project name under `mflab/Projects` Then create a new folder `cases` in that new project folder and then create a new case by creating a new specific case pertaining to that project. One can have an unlimited number of cases under each project. In the new case folder copy the directories of another case and empty them except for the four files mentioined above to allow a fresh start. Rename the workbook to `<case_name>.xlsx`. Then adapt the four files `mf_adapt.py`, `settings.py` and `mf_analyze.py` to represent the new case.

After `Modflow` etc. have (successfully finished), `mf_analyze.py` is run to show the results.

The user should not have to changeneither `mf_run.py` nor `mf_setup.py`.

@TO251119

# Imports

mf6_bootstrap puts the paths of the relevant source directories into sys.path.

1) mf6lab/src
2) mf6lab/Projects/<project>/src
3) mf6lab/Projects/<project>/cases/<case>/src

logging_setup sets up logging.basic_configuration()

For this to work, the file has to be launched form a case or `case/subdir` directory.

In [ ]:
import mf6_bootstrap # noqa: F401
import flopy
import logging
import mf6tools
import mf_adapt
import logging_setup # noqa F401

# --- setting up the logger
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

In [ ]:

gr = mf_adapt.gr

def mf_setup():

    #params_wbk = os.path.join(mf_adapt.HOME, mf_adapt.sim_name + '.xlsx')

    use_models, use_packages  = mf6tools.get_models_and_packages_from_excel(
                                                mf_adapt.params_wbk, sheet_name='NAM')
    
    model_dict = mf6tools.get_mf6_params_from_excel(
                                    mf_adapt.params_wbk, sheet_name='SIM6')

    if 'Gwf' in use_models:
        model_dict.update(mf6tools.get_mf6_params_from_excel(
                                    mf_adapt.params_wbk, sheet_name='GWF6'))
    if 'Gwt' in use_models:
        model_dict.update(mf6tools.get_mf6_params_from_excel(
                                    mf_adapt.params_wbk, sheet_name='GWT6'))

    fp_packages=dict()

### === S I M U L A T I O N =========================================

### sim =====================
    logging.info("sim")
    model_dict['Simsim'].update(sim_name=mf_adapt.sim_name, sim_ws=mf_adapt.dirs.SIM)
    sim = flopy.mf6.MFSimulation(**model_dict['Simsim'])
    fp_packages['Simsim'] = sim

### tdis ====================
    logging.info("Simtdis")
    model_dict['Simtdis'].update(**mf_adapt.Simtdis)
    fp_packages['Simtdis'] = flopy.mf6.ModflowTdis(sim, **model_dict['Simtdis'])
            
### ======== F L O W ================================================

    if 'Gwf' in use_models:
    ### Gwf =====================
        if True: # Groundwater flow model, use name from 'sim').
            logging.info("Gwf")
            Gwf_model_name = sim.name + 'Gwf'
            model_dict['Gwfgwf'].update(modelname=Gwf_model_name,
                                    model_rel_path=mf_adapt.dirs.GWF,
                                    exe_name=sim.exe_name,
                                    )
            gwf = flopy.mf6.ModflowGwf(sim, **model_dict['Gwfgwf'])
            fp_packages['Gwfgwf'] = gwf
            
            ### Ims =====================
            model_dict['Gwfims'].update()
            logging.info("Gwfims")
            Gwfims = flopy.mf6.ModflowIms(sim, **model_dict['Gwfims'])
            fp_packages['Gwfims'] = Gwfims
            sim.register_ims_package(Gwfims, [Gwf_model_name])    
    

    ### Gwfdis ==================
        if 'Gwfdis' in use_packages:
            logging.info('Gwfdis')
            model_dict['Gwfdis'].update(**mf_adapt.Gwfdis)
            gr = model_dict['Gwfdis'].pop('gr')
            model_dict['Gwfdis'].update(nlay=gr.nlay, nrow=gr.nrow, ncol=gr.ncol,
                                        delr=gr.dx, delc=gr.dy,
                                        top=gr.Z[0], botm=gr.Z[1:]                              
            )
            fp_packages['Gwfdis'] = flopy.mf6.ModflowGwfdis(gwf, **model_dict['Gwfdis'])

    ### Gwfdisu ==================
        if 'Gwfdisu' in use_packages:
            logging.info('Gwfdisu')
            model_dict['Gwfdisu'].update(**mf_adapt.Gwfdisu)
            fp_packages['Gwfdisu'] = flopy.mf6.ModflowGwfdisu(gwf, **model_dict['Gwfdisu'])
    
    ### Gwfdisv ==================
        if 'Gwfdisv' in use_packages:
            logging.info('Gwfdisv')
            model_dict['Gwfdisv'].update(**mf_adapt.Gwfdisv)
            fp_packages['Gwfdisv'] = flopy.mf6.ModflowGwfdisv(gwf, **model_dict['Gwfdisv'])
    
    
    ### Gwfbuy ==================
        if 'Gwfbuy' in use_packages:
            logging.info('Gwfbuy')
            model_dict['Gwfbuy'].update(**mf_adapt.Gwfbuy)
            fp_packages['Gwfbuy'] = flopy.mf6.ModflowGwfbuy(gwf, **model_dict['Gwfbuy'])
    
    ### Gwfchd ==================
        if 'Gwfchd' in use_packages:
            logging.info('Gwfchd')
            timeseries = mf_adapt.Gwfchd.pop('timeseries', None)
            model_dict['Gwfchd'].update(**mf_adapt.Gwfchd)
            fp_packages['Gwfchd'] = flopy.mf6.ModflowGwfchd(gwf, **model_dict['Gwfchd'])
            Gwfchd = fp_packages['Gwfchd']
            if timeseries:
                Gwfchd.ts.initialize(**timeseries.pop(0))
                while timeseries:
                    Gwfchd.ts.append_package(**timeseries.pop(0))
    
    ### Gwfcsub ==================
        if 'Gwfcsub' in use_packages:
            logging.info('Gwfcsub')
            model_dict['Gwfcsub'].update(**mf_adapt.Gwfcsub)
            fp_packages['Gwfcsub'] = flopy.mf6.ModflowGwfcsub(gwf, **model_dict['Gwfcsub'])
    
    ### Gwfdrn ==================
        if 'Gwfdrn' in use_packages:
            logging.info('Gwfdrn')
            model_dict['Gwfdrn'].update(**mf_adapt.Gwfdrn)
            fp_packages['Gwfdrn'] = flopy.mf6.ModflowGwfdrn(gwf, **model_dict['Gwfdrn'])
    
    ### Gwfevt ================== list-based input
        if 'Gwfevt' in use_packages:
            logging.info('Gwfevt')
            model_dict['Gwfevt'].update(**mf_adapt.Gwfevt)
            fp_packages['Gwfevt'] = flopy.mf6.ModflowGwfevt(gwf, **model_dict['Gwfevt'])
    
    ### Gwfevta ================== array-based input
        if 'Gwfevta' in use_packages:
            logging.info('Gwfevta')
            model_dict['Gwfevta'].update(**mf_adapt.Gwfevta)
            fp_packages['Gwfevta'] = flopy.mf6.ModflowGwfevta(gwf, **model_dict['Gwfevta'])
    
    ### Gwfghb ==================
        if 'Gwfghb' in use_packages:
            logging.info('Gwfghb')
            model_dict['Gwfghb'].update(**mf_adapt.Gwfghb)
            fp_packages['Gwfghb'] = flopy.mf6.ModflowGwfghb(gwf, **model_dict['Gwfghb'])
    
    ### Gwfgnc ==================
        if 'Gwfgnc' in use_packages:
            logging.info('Gwfgnc')
            model_dict['Gwfgnc'].update(**mf_adapt.Gwfgnc)
            fp_packages['Gwfgnc'] = flopy.mf6.ModflowGwfgnc(gwf, **model_dict['Gwfgnc'])
    
    ### Gwfhfb ==================
        if 'Gwfhfb' in use_packages:
            logging.info('Gwfhfb')
            model_dict['Gwfhfb'].update(**mf_adapt.Gwfhfb)
            fp_packages['Gwfhfb'] = flopy.mf6.ModflowGwfhfb(gwf, **model_dict['Gwfhfb'])
    
    ### Gwfic ==================
        if 'Gwfic' in use_packages:
            logging.info('Gwfic')
            model_dict['Gwfic'].update(**mf_adapt.Gwfic)
            fp_packages['Gwfic'] = flopy.mf6.ModflowGwfic(gwf, **model_dict['Gwfic'])
    
    ### Gwflak ==================
        if 'Gwflak' in use_packages:
            logging.info('Gwflak')
            model_dict['Gwflak'].update(**mf_adapt.Gwflak)
            fp_packages['Gwflak'] = flopy.mf6.ModflowGwflak(gwf, **model_dict['Gwflak'])
    
    ### Gwfmaw ==================
        if 'Gwfmaw' in use_packages:
            logging.info('Gwfmaw')
            model_dict['Gwfmaw'].update(**mf_adapt.Gwfmaw)
            fp_packages['Gwfmaw'] = flopy.mf6.ModflowGwfmaw(gwf, **model_dict['Gwfmaw'])
    
    ### Gwfmvr ==================
        if 'Gwfmvr' in use_packages:
            logging.info('Gwfmvr')
            model_dict['Gwfmvr'].update(mf_adapt.Gwfmvr)
            fp_packages['Gwfmvr'] = flopy.mf6.ModflowGwfmvr(gwf, **model_dict['Gwfmvr'])
    
    ### Gwfnpf ==================
        if 'Gwfnpf' in use_packages:
            logging.info('Gwfnpf')
            model_dict['Gwfnpf'].update(**mf_adapt.Gwfnpf)
            fp_packages['Gwfnpf'] = flopy.mf6.ModflowGwfnpf(gwf, **model_dict['Gwfnpf'])
    
    ### Gwfobs ==================
        if 'Gwfobs' in use_packages:
            logging.info('Gwfobs')
            model_dict['Gwfobs'].update(**mf_adapt.Gwfobs)
            fp_packages['Gwfobs'] = flopy.mf6.ModflowGwfobs(gwf, **model_dict['Gwfobs'])
    
    ### Gwfoc ==================
        if 'Gwfoc' in use_packages:
            logging.info('Gwfoc')
            model_dict['Gwfoc'].update(**mf_adapt.Gwfoc)
            fp_packages['Gwfoc'] = flopy.mf6.ModflowGwfoc(gwf, **model_dict['Gwfoc'])
    
    ### Gwfrch ==================
        if 'Gwfrch' in use_packages:
            logging.info('Gwfrch')
            model_dict['Gwfrch'].update(**mf_adapt.Gwfrch)
            fp_packages['Gwfrch'] = flopy.mf6.ModflowGwfrch(gwf, **model_dict['Gwfrch'])
    
    ### Gwfrcha ==================
        if 'Gwfrcha' in use_packages:
            logging.info('Gwfrcha')
            model_dict['Gwfrcha'].update(**mf_adapt.Gwfrcha)
            fp_packages['Gwfrcha'] = flopy.mf6.ModflowGwfrcha(gwf, **model_dict['Gwfrcha'])
    
    ### Gwfriv ==================
        if 'Gwfriv' in use_packages:
            logging.info('Gwfriv')
            model_dict['Gwfriv'].update(**mf_adapt.Gwfriv)
            fp_packages['Gwfriv'] = flopy.mf6.ModflowGwfriv(gwf, **model_dict['Gwfriv'])
    
    ### Gwfsfr ==================
        if 'Gwfsfr' in use_packages:
            logging.info('Gwfsfr')
            model_dict['Gwfsfr'].update(mf_adapt.Gwfsfr)
            fp_packages['Gwfsfr'] = flopy.mf6.ModflowGwfsfr(gwf, **model_dict['Gwfsfr'])
    
    ### Gwfsto ==================
        if 'Gwfsto' in use_packages:
            logging.info('Gwfsto')
            model_dict['Gwfsto'].update(**mf_adapt.Gwfsto)
            fp_packages['Gwfsto'] = flopy.mf6.ModflowGwfsto(gwf, **model_dict['Gwfsto'])
    
    ### Gwfuzf ==================
        if 'Gwfuzf' in use_packages:
            logging.info('Gwfuzf')
            model_dict['Gwfuzf'].update(**mf_adapt.Gwfuzf)
            fp_packages['Gwfuzf'] = flopy.mf6.ModflowGwfuzf(gwf, **model_dict['Gwfuzf'])
    
    ### Gwfvsc ==================
        if 'Gwfvsc' in use_packages:
            logging.info('Gwfvsc')
            model_dict['Gwfvsc'].update(**mf_adapt.Gwfvsc)
            fp_packages['Gwfvsc'] = flopy.mf6.ModflowGwfvsc(gwf, **model_dict['Gwfvsc'])
    
    ### Gwfwel ==================
        if 'Gwfwel' in use_packages:
            logging.info('Gwfwel')
            model_dict['Gwfwel'].update(**mf_adapt.Gwfwel)
            fp_packages['Gwfwel'] = flopy.mf6.ModflowGwfwel(gwf, **model_dict['Gwfwel'])


    ### ===== T R A N S P O R T =================================================

    if 'Gwt' in use_models:
    ### Gwt ===============================
        if True: # Groundwater transport model, use name from 'sim').
            logging.info("Gwtgwt")
            Gwt_model_name = sim.name + 'Gwt'
            model_dict['Gwtgwt'].update(modelname=Gwt_model_name,
                                    model_rel_path=mf_adapt.dirs.GWT,
                                    exe_name=sim.exe_name,
                                    )
            gwt = flopy.mf6.ModflowGwt(sim, **model_dict['Gwtgwt'])
            fp_packages['Gwtgwt'] = gwt
            
            ### ims =====================
            model_dict['Gwtims'].update(complexity='MODERATE')  # SIMPLE | MODERATE | COMPLEX
            logging.info("Gwtims")
            Gwtims = flopy.mf6.ModflowIms(sim, **model_dict['Gwtims'])
            fp_packages['Gwtims'] = Gwtims
            sim.register_ims_package(Gwtims, [Gwt_model_name])
            
    ### Gwtdis ==================
        if 'Gwtdis' in use_packages:
            logging.info('Gwtdis')            
            model_dict['Gwtdis'].update(**mf_adapt.Gwfdis)
            gr = model_dict['Gwtdis'].pop('gr')
            model_dict['Gwtdis'].update(nlay=gr.nlay, nrow=gr.nrow, ncol=gr.ncol,
                                        delr=gr.dx, delc=gr.dy,
                                        top=gr.Z[0], botm=gr.Z[1:]                              
            )
            fp_packages['Gwtdis'] = flopy.mf6.ModflowGwtdis(gwt, **model_dict['Gwtdis'])
    
    ### Gwtdisu ==================
        if 'Gwtdisu' in use_packages:
            logging.info('Gwtdisu')
            model_dict['Gwtdisu'].update(**mf_adapt.Gwtdisu)
            fp_packages['Gwtdisu'] = flopy.mf6.ModflowGwtdisu(gwt, **model_dict['Gwtdisu'])
    
    ### Gwtdisv ==================
        if 'Gwtdisv' in use_packages:
            logging.info('Gwtdisv')
            model_dict['Gwtdisv'].update(**mf_adapt.Gwtdisv)
            fp_packages['Gwtdisv'] = flopy.mf6.ModflowGwtdisv(gwt, **model_dict['Gwtdisv'])
            
    ### Gwtadv ==================
        if 'Gwtadv' in use_packages:
            logging.info('Gwtadv')
            model_dict['Gwtadv'].update(**mf_adapt.Gwtadv)
            fp_packages['Gwtadv'] = flopy.mf6.ModflowGwtadv(gwt, **model_dict['Gwtadv'])
    
    ### Gwtcnc ==================
        if 'Gwtcnc' in use_packages:
            logging.info('Gwtcnc')
            model_dict['Gwtcnc'].update(**mf_adapt.Gwtcnc)
            fp_packages['Gwtcnc'] = flopy.mf6.ModflowGwtcnc(gwt, **model_dict['Gwtcnc'])
    
    ### Gwtdsp ==================
        if 'Gwtdsp' in use_packages:
            logging.info('Gwtdsp')
            model_dict['Gwtdsp'].update(**mf_adapt.Gwtdsp)
            fp_packages['Gwtdsp'] = flopy.mf6.ModflowGwtdsp(gwt, **model_dict['Gwtdsp'])
    
    ### Gwtfmi ================== Flow model interface (for using data from previously run flow model)
        if 'Gwtfmi' in use_packages:
            logging.info('Gwtfmi')
            model_dict['Gwtfmi'].update(**mf_adapt.Gwtfmi)
            fp_packages['Gwtfmi'] = flopy.mf6.ModflowGwtfmi(gwt, **model_dict['Gwtfmi'])
    
    ### Gwtic ==================
        if 'Gwtic' in use_packages:
            logging.info('Gwtic')
            model_dict['Gwtic'].update(**mf_adapt.Gwtic)
            fp_packages['Gwtic'] = flopy.mf6.ModflowGwtic(gwt, **model_dict['Gwtic'])
    
    ### Gwtist ==================
        if 'Gwtist' in use_packages:
            logging.info('Gwtist')
            model_dict['Gwtist'].update(**mf_adapt.Gwtist)
            fp_packages['Gwtist'] = flopy.mf6.ModflowGwtist(gwt, **model_dict['Gwtist'])
    
    ### Gwtlkt ==================
        if 'Gwtlkt' in use_packages:
            logging.info('Gwtlkt')
            model_dict['Gwtlkt'].update(**mf_adapt.Gwtlkt)
            fp_packages['Gwtlkt'] = flopy.mf6.ModflowGwtlkt(gwt, **model_dict['Gwtlkt'])
    
    ### Gwtmst ==================
        if 'Gwtmst' in use_packages:
            logging.info('Gwtmst')
            model_dict['Gwtmst'].update(**mf_adapt.Gwtmst)
            fp_packages['Gwtmst'] = flopy.mf6.ModflowGwtmst(gwt, **model_dict['Gwtmst'])
    
    ### Gwtmvt ==================
        if 'Gwtmvt' in use_packages:
            logging.info('Gwtmvt')
            model_dict['Gwtmvt'].update(**mf_adapt.Gwtmvt)
            fp_packages['Gwtmvt'] = flopy.mf6.ModflowGwtmvt(gwt, **model_dict['Gwtmvt'])
    
    ### Gwtmwt ==================
        if 'Gwtmwt' in use_packages:
            logging.info('Gwtmwt')
            model_dict['Gwtmwt'].update(**mf_adapt.Gwtmwt)
            fp_packages['Gwtmwt'] = flopy.mf6.ModflowGwtmwt(gwt, **model_dict['Gwtmwt'])
    
    ### Gwtobs ==================
        if 'Gwtobs' in use_packages:
            logging.info('Gwtobs')
            model_dict['Gwtobs'].update(**mf_adapt.Gwtobs)
            fp_packages['Gwtobs'] = flopy.mf6.ModflowGwtobs(gwt, **model_dict['Gwtobs'])
    
    ### Gwtoc ==================
        if 'Gwtoc' in use_packages:
            logging.info('Gwtoc')
            model_dict['Gwtoc'].update(**mf_adapt.Gwtoc)
            fp_packages['Gwtoc'] = flopy.mf6.ModflowGwtoc(gwt, **model_dict['Gwtoc'])
    
    ### Gwtsft ==================
        if 'Gwtsft' in use_packages:
            logging.info('Gwtsft')
            model_dict['Gwtsft'].update(**mf_adapt.Gwtsft)
            fp_packages['Gwtsft'] = flopy.mf6.ModflowGwtsft(gwt, **model_dict['Gwtsft'])
    
    ### Gwtsrc ==================
        if 'Gwtsrc' in use_packages:
            logging.info('Gwtsrc')
            model_dict['Gwtsrc'].update(**mf_adapt.Gwtsrc)
            fp_packages['Gwtsrc'] = flopy.mf6.ModflowGwtsrc(gwt, **model_dict['Gwtsrc'])
    
    ### Gwtssm ==================
        if 'Gwtssm' in use_packages:
            logging.info('Gwtssm')
            model_dict['Gwtssm'].update(**mf_adapt.Gwtssm)
            fp_packages['Gwtssm'] = flopy.mf6.ModflowGwtssm(gwt, **model_dict['Gwtssm'])
    
    ### Gwtuzt ==================
        if 'Gwtuzt' in use_packages:
            logging.info('Gwtuzt')
            model_dict['Gwtuzt'].update(**mf_adapt.Gwtuzt)
            fp_packages['Gwtuzt'] = flopy.mf6.ModflowGwtuzt(gwt, **model_dict['Gwtuzt'])



    ### ==== Dynamic exchange between GWF and GWT model ===========================================

    # This dynamic exchange is automatic:
    # Gwf model is on when any of the Gwf packages are on.
    # Gwt model is on when any of the GWt packages are on.
    # Dynamic exchange is on when both Gwf and Gwt model are on.
     
    if 'Gwf' in use_models and 'Gwt' in use_models:
        logging.info("Dynamic exchange GWF-GWT active")
        model_dict['Gwfexc'].update(exgtype='GWF6-GWT6',
                                    exgmnamea=model_dict['Gwfgwf']['modelname'],
                                    exgmnameb=model_dict['Gwtgwt']['modelname'])
        fp_packages['Gwfexc'] = flopy.mf6.ModflowGwfgwt(sim, **model_dict['Gwfexc'])

    # return fp_packages, model_dict, use_models, use_packages


    ### === Modpath 7 ===================================================
    # TODO: TO251119 Look at exactly how to deal with MP7 which is is not included in mf6!
    # TODO: I used MP7 in the dispersion project, see how it was done there.

    if 'MP7' in use_models:
    ### MP7 =====================
        if True: # Groundwater path model, use name from 'sim').
            logging.info("MP7")
            MP7_model_name = sim.name + 'MP7'
            model_dict['MP7mp7'].update(modelname=MP7_model_name,
                                    model_rel_path=mf_adapt.dirs.MP7,
                                    exe_name=sim.exe_name,
                                    )
            mp7 = flopy.modpath.Modpath7.create_mp7(sim, **model_dict['Gwfgwf'])
            fp_packages['MP7mp7'] = mp7
            
    ### Ims =====================
            model_dict['Gwfims'].update()
            logging.info("Gwfims")
            Gwfims = flopy.mf6.ModflowIms(sim, **model_dict['Gwfims'])
            fp_packages['Gwfims'] = Gwfims
            sim.register_ims_package(Gwfims, [Gwf_model_name])    


    ### mp7 ==================
        if 'MP7' in use_packages:
            logging.info('MP7')
            model_dict['MP7'].update(**mf_adapt.MP7)
            fp_packages['MP7'] = flopy.modpath.Modpath7.create_mp7(mp7, **model_dict['Gwtsrc'])
            
    return fp_packages, model_dict, use_models, use_packages



def run_modflow(sim=None):
    """Simulate GGOR using MODFLOW.

    Parameters
    ----------
    sim: flopy.mf6.MFSimulation object completely defines the model.
    """
    # Write simulation
    sim.write_simulation(silent=mf_adapt.user_settings['silent'])
    
    # Run simulation
    success, buff = sim.run_simulation(silent=mf_adapt.user_settings['silent'])

    print('Running success = {}'.format(success))
    if not success:
        print(buff)
        logging.critical("Buffer printed because MODFLOW did not terminate normally.")
        raise Exception('MODFLOW did not terminate normally.')
    return success



if __name__ == '__main__':
    
    fp_packages, model_dict, use_models, use_packages = mf_setup()
    
    success = run_modflow(sim=fp_packages['sim'])
        
    print(mf_adapt.sim_name)
    